In [2]:
import sys
sys.path.append('../')

In [3]:

from datasets import load_dataset
def load_xsum():
    print("Loading XSum dataset...")
    # Load the dataset from the specified path
    dataset = load_dataset("EdinburghNLP/xsum", split="train")
    return dataset

# load_xsum()

/mnt/hdd/git/LLM-detection/when-human-write-like-machines/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:

def load_owt():
    print("Loading Owt dataset...")
    # Load the dataset from the specified path
    dataset = load_dataset("Skylion007/openwebtext", split="train")
    return dataset


In [9]:
load_owt()

Loading Owt dataset...


KeyboardInterrupt: 

In [11]:
def load_wp():
    print("Loading Writing Prompts dataset...")
    # Load the dataset from the specified path
    dataset = load_dataset("llm-aes/writing-prompts", split="train")
    return dataset


load_wp()

Loading Writing Prompts dataset...


Dataset({
    features: ['prompt'],
    num_rows: 232360
})

# notoxic

In [3]:
from dataset import get_notoxic_dataset

dataset =get_notoxic_dataset('xsum-processed')

/mnt/hdd/git/LLM-detection/when-human-write-like-machines/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from text_filter import StructuralArtifactFilter, TextWordLengthTruncator

word_length_truncator = TextWordLengthTruncator(min_length=150, max_length=300)
artifact_filter_truncator = StructuralArtifactFilter()

word_length_dataset = dataset.filter(word_length_truncator.filter, batched=True, batch_size=64, load_from_cache_file=False)
new_dataset = word_length_dataset.filter(artifact_filter_truncator.filter, batched=True, batch_size=64, load_from_cache_file=False)


Filter: 100%|██████████| 35623/35623 [00:07<00:00, 4912.37 examples/s]


In [5]:
new_dataset = new_dataset.select(range(100))

In [8]:
from text_filter import NotAcceptableFilter


na_filter = NotAcceptableFilter()
na_dataset = new_dataset.filter(na_filter.filter, batched=True, batch_size=64, load_from_cache_file=False)

Loading quantized model with Outlines...


Filter: 100%|██████████| 100/100 [00:20<00:00,  4.87 examples/s]


In [9]:
na_dataset

Dataset({
    features: ['text'],
    num_rows: 99
})

# WP


In [4]:
import sys
sys.path.append('../')
from utils.dataset import DatasetLoader


dataset_loader = DatasetLoader("~/.datasets")
dataset_loader.load("wp-processed_notoxic")


In [ ]:
from utils.text_filter import TextWordLengthTruncator
word_length_filter = TextWordLengthTruncator(
            min_length=150,
            max_length=500
)

In [9]:

dataset_loader.dataset.filter(word_length_filter.filter, batched=True, batch_size=64, load_from_cache_file=False)

Filter: 100%|██████████| 232360/232360 [00:00<00:00, 871374.20 examples/s]


Dataset({
    features: ['text'],
    num_rows: 0
})

In [12]:
from datasets import load_dataset

dataset = load_dataset("euclaise/writingprompts", split="train")

Generating validation split: 100%|██████████| 15620/15620 [00:00<00:00, 260447.42 examples/s]


In [13]:
dataset

Dataset({
    features: ['prompt', 'story'],
    num_rows: 272600
})

In [14]:
dataset = load_dataset("EdinburghNLP/xsum", split="train")

In [16]:
dataset['document']

Column(['The full cost of damage in Newton Stewart, one of the areas worst affected, is still being assessed.\nRepair work is ongoing in Hawick and many roads in Peeblesshire remain badly affected by standing water.\nTrains on the west coast mainline face disruption due to damage at the Lamington Viaduct.\nMany businesses and householders were affected by flooding in Newton Stewart after the River Cree overflowed into the town.\nFirst Minister Nicola Sturgeon visited the area to inspect the damage.\nThe waters breached a retaining wall, flooding many commercial properties on Victoria Street - the main shopping thoroughfare.\nJeanette Tate, who owns the Cinnamon Cafe which was badly affected, said she could not fault the multi-agency response once the flood hit.\nHowever, she said more preventative work could have been carried out to ensure the retaining wall did not fail.\n"It is difficult but I do think there is so much publicity for Dumfries and the Nith - and I totally appreciate th

In [1]:
import sys
sys.path.append('../')

In [5]:
from hydra import compose, initialize
from omegaconf import OmegaConf

with initialize(version_base=None, config_path="../conf"):
    cfg = compose(config_name="config")

print(cfg.experiment.ranges)

{'short': {'range': [150, 220], 'n_per_domain': 200}, 'medium': {'range': [221, 350], 'n_per_domain': 200}, 'long': {'range': [351, 500], 'n_per_domain': 200}}


In [31]:
from datasets import load_dataset, load_from_disk
# dataset = load_from_disk("~/.datasets/xsum-processed_notoxic")
dataset = load_dataset("euclaise/writingprompts", split="train")

In [33]:
from utils.range_classification import add_range_to_dataset

dataset = dataset.rename_column("story", "text")

range_dataset = add_range_to_dataset(dataset, 
                     cfg.experiment.ranges.short.range, 
                     cfg.experiment.ranges.medium.range, 
                        cfg.experiment.ranges.long.range)

Map: 100%|██████████| 272600/272600 [00:08<00:00, 31941.63 examples/s]


In [34]:
from utils.sampling import stratified_random_sampling_by_range



sampled_dataset = stratified_random_sampling_by_range(range_dataset,
                                                      cfg.experiment.ranges)

Filter: 100%|██████████| 272600/272600 [00:00<00:00, 416358.12 examples/s]


In [35]:
sampled_dataset

Dataset({
    features: ['prompt', 'text', 'length', 'range'],
    num_rows: 600
})

In [36]:
from utils.dataset import save_texts_line_by_line


save_texts_line_by_line(sampled_dataset, "wp_sampled.txt", text_column="text")

In [37]:
from datasets import load_dataset, Dataset

dataset_stream = load_dataset(
    "Skylion007/openwebtext",
    split="train",
    streaming=True
)

dataset_500k = Dataset.from_list(
    list(dataset_stream.take(500_000))
)


In [39]:
from utils.range_classification import add_range_to_dataset


range_dataset = add_range_to_dataset(dataset_500k, 
                     cfg.experiment.ranges.short.range, 
                     cfg.experiment.ranges.medium.range, 
                        cfg.experiment.ranges.long.range)

Map:   0%|          | 0/500000 [00:00<?, ? examples/s]

Map: 100%|██████████| 500000/500000 [00:21<00:00, 22749.20 examples/s]


In [40]:
sampled_dataset = stratified_random_sampling_by_range(range_dataset,
                                                      cfg.experiment.ranges)

Filter: 100%|██████████| 500000/500000 [00:01<00:00, 346753.04 examples/s]


In [ ]:
from utils.dataset import save_texts_line_by_line


save_texts_line_by_line(sampled_dataset, "owt_sampled.txt", text_column="text")